# Notebook 14 – Scikit-learn Preprocessing Pipeline
## Why Use Pipelines?

Across this series, we've been cleaning, imputing, scaling, and encoding
columns one line at a time, and manually remembering to fit only on
`train_df` and apply to `test_df` (Notebook 12 and 13's whole lesson).

A **Pipeline** bundles all of that into a single object that:

* Runs every step in the correct order automatically.
* Guarantees `.fit()` only ever happens on training data, since you call
  it once on `train_df` and reuse the same fitted pipeline everywhere else.
* Prevents the exact leakage mistakes we deliberately caused in
  Notebook 13, structurally, not just by remembering to be careful.
* Makes the whole preprocessing recipe reusable and easy to hand off.

Let's build one using our real numeric and categorical columns.

In [1]:
import pandas as pd

df = pd.read_csv("hr_employee_attrition_raw.csv")
df["Attrition_binary"] = df["Attrition"].map({"Yes": 1, "No": 0})

# Light cleanup so the pipeline receives genuinely numeric/text columns
df["MonthlyIncome"] = pd.to_numeric(
    df["MonthlyIncome"].astype(str).str.replace(" USD", "", regex=False), errors="coerce"
)
df["YearsAtCompany"] = pd.to_numeric(
    df["YearsAtCompany"].astype(str).str.replace(" yrs", "", regex=False), errors="coerce"
)

df.shape

(1230, 17)

In [2]:
from sklearn.model_selection import train_test_split

numeric_features = ["Age", "MonthlyIncome", "YearsAtCompany", "DistanceFromHomeKM",
                      "JobSatisfaction", "PerformanceRating"]
categorical_features = ["Gender", "Department", "JobRole", "Education", "OverTime"]

X = df[numeric_features + categorical_features]
y = df["Attrition_binary"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

## Numerical Pipeline: Imputation → Scaling

Each numeric column gets two steps, in order:

1. **Imputation**: fill missing values. We saw missing values in
   `PerformanceRating` and elsewhere back in Notebook 5, using the median
   is a safe default since it's resistant to the outliers we found in
   Notebook 6 (like `Age` = 999, `DistanceFromHomeKM` = 830).
2. **Scaling**: `StandardScaler`, so all numeric columns sit on a
   comparable range, as covered in Notebook 8.

`Pipeline` chains these so both steps run in sequence automatically.

In [3]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

## Categorical Pipeline: Imputation → Encoding

Categorical columns need their own two steps:

1. **Imputation**: fill missing values (like the blank `Education` entries
   we saw in Notebook 5) with a constant placeholder, since there's no
   meaningful "average" category.
2. **Encoding**: `OneHotEncoder`, matching Notebook 7's guidance for
   nominal, low-to-medium cardinality columns like `Department` and
   `Gender`. `handle_unknown="ignore"` protects against the unseen-category
   problem also covered in Notebook 7, any category not seen in training
   just becomes all zeros instead of crashing.

In [4]:
from sklearn.preprocessing import OneHotEncoder

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

## ColumnTransformer: Combining Both Pipelines

`ColumnTransformer` applies the right pipeline to the right columns, all
in one step. Numeric columns go through the numeric pipeline, categorical
columns go through the categorical pipeline, and the results are combined
back into a single feature matrix.

In [5]:
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

## `fit()`, `transform()`, and `fit_transform()`

Three related methods, each with a specific, deliberate use:

* **`fit()`**: learns the necessary statistics (median for imputation,
  mean/std for scaling, category list for encoding) from the data it's
  given. Should only ever be called on `X_train`.
* **`transform()`**: applies whatever was already learned during `fit()`
  to new data, without recalculating anything. This is what gets called
  on `X_test`.
* **`fit_transform()`**: a shortcut that does both at once, `fit()`
  followed by `transform()`, purely for convenience on the training set.

Using `fit_transform()` on the test set would be the exact leakage mistake
from Notebook 13, it would relearn statistics using test data instead of
applying what was learned from training.

In [6]:
# fit_transform on TRAINING data: learn AND apply in one step
X_train_processed = preprocessor.fit_transform(X_train)

# transform ONLY on test data: apply what was already learned, never refit
X_test_processed = preprocessor.transform(X_test)

print("Training set shape after preprocessing:", X_train_processed.shape)
print("Test set shape after preprocessing:", X_test_processed.shape)

Training set shape after preprocessing: (984, 46)
Test set shape after preprocessing: (246, 46)


## Inspecting What the Pipeline Actually Built

Worth checking what column names came out the other side, especially
after one-hot encoding expands categorical columns into several.

In [7]:
feature_names = preprocessor.get_feature_names_out()
print("Total features after preprocessing:", len(feature_names))
print(feature_names[:15])

Total features after preprocessing: 46
['num__Age' 'num__MonthlyIncome' 'num__YearsAtCompany'
 'num__DistanceFromHomeKM' 'num__JobSatisfaction' 'num__PerformanceRating'
 'cat__Gender_F' 'cat__Gender_FEMALE' 'cat__Gender_Female' 'cat__Gender_M'
 'cat__Gender_Male' 'cat__Gender_Missing' 'cat__Gender_male'
 'cat__Department_Engineering' 'cat__Department_Finance']


## Using the Pipeline End-to-End with a Model

The real benefit shows up here, the preprocessor can be chained directly
into a full pipeline with a model, so calling `.fit()` once handles
everything, cleaning, scaling, encoding, and model training together.

In [9]:
from sklearn.linear_model import LogisticRegression

full_pipeline = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

# One call trains the entire pipeline correctly, in the right order
full_pipeline.fit(X_train, y_train)

train_accuracy = round(full_pipeline.score(X_train, y_train), 3)
test_accuracy = round(full_pipeline.score(X_test, y_test), 3)

print("Training accuracy:", train_accuracy)
print("Test accuracy:", test_accuracy)

Training accuracy: 0.598
Test accuracy: 0.537


## Why This Structure Prevents Leakage Automatically

Compare this to how we built things manually in earlier notebooks:

* No separate `scaler.fit()` call floating around that could accidentally
  run on the full dataset (Notebook 12's mistake).
* No manually remembering to apply `.map()` for target encoding without
  refitting (Notebook 7's mistake, Notebook 13's leaky example).
* Calling `full_pipeline.fit(X_train, y_train)` only ever touches training
  data, and `full_pipeline.predict(X_test)` or `.score(X_test, y_test)`
  automatically calls `.transform()` internally, never `.fit()` again.

The pipeline structure itself enforces the rule, rather than relying on
the person writing the code to remember it every single time.

Structure of what we built:

* **Numeric path**: `Age`, `MonthlyIncome`, `YearsAtCompany`,
  `DistanceFromHomeKM`, `JobSatisfaction`, `PerformanceRating` →
  median imputation → standard scaling.
* **Categorical path**: `Gender`, `Department`, `JobRole`, `Education`,
  `OverTime` → constant imputation → one-hot encoding.
* Both paths combined through `ColumnTransformer`, then chained into a
  full model pipeline with a single `.fit()` call.